<a href="https://colab.research.google.com/github/clee2026/MSDS_498_Capstone/blob/main/init_findings/notebook_2_Prepare_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook 2 - Data Conditioning and Feature Finalization (20M Colab Version)

## Required Inputs

Upload the following files into Colab:

### From Notebook 1 Outputs
- column_profile.csv
- missingness_summary.csv
- candidate_drop_review.csv
- run_manifest.json

### Notes
- These files are generated from Notebook 1.
- They are used to determine which fields to keep, drop, and transform.
- The run_manifest.json file contains the source parquet path and the original output paths from Notebook 1.
- This notebook is built to work whether those files are still in their original /content/project_outputs/... folders or were uploaded directly into /content/.

## Pipeline Context

This notebook is part of a 3-step pipeline:

1. Notebook 1 — Data Profiling
2. Notebook 2 — Data Conditioning and Feature Engineering
3. Notebook 3 — Analysis and Modeling

Each step depends on the outputs of the previous step.

## What this notebook does

This notebook will:

- load the Notebook 1 handoff files
- load only the selected columns from the raw parquet
- handle missing values in a memory-conscious way
- standardize key fields
- create engineered features
- save a full prepared parquet to Google Drive
- save a local sample parquet for Notebook 3
- zip the local Notebook 2 outputs for download


In [ ]:
# 1. Environment Setup

import os
import json
import shutil
import warnings
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive



## Load manifest and prepare output folders

This section loads the Notebook 1 manifest and creates the Notebook 2 output directories.


In [ ]:
# 2. Load Manifest and Prepare Paths

MANIFEST_FALLBACK = "/content/run_manifest.json"
if not os.path.exists(MANIFEST_FALLBACK):
    raise FileNotFoundError(
        "run_manifest.json was not found in /content. Upload it before running Notebook 2."
    )

with open(MANIFEST_FALLBACK, "r") as f:
    manifest = json.load(f)

SOURCE_PARQUET = manifest["source_parquet"]
OUTPUT_DIR = manifest.get("output_dir", "/content/project_outputs")

COND_DIR = f"{OUTPUT_DIR}/02_conditioning"
TABLE_DIR = f"{COND_DIR}/tables"
PLOT_DIR = f"{COND_DIR}/plots"

os.makedirs(COND_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print("Source parquet:", SOURCE_PARQUET)
print("Notebook 2 output dir:", COND_DIR)


Source parquet: /content/drive/MyDrive/project_data/Capstone Course Project Files/Data/nyc_311_FINAL_UPDATED.parquet
Notebook 2 output dir: /content/project_outputs/02_conditioning



## Load Notebook 1 outputs with path fallback support

The manifest contains the original output paths from Notebook 1.  
If those paths do not exist in the current Colab session, this notebook automatically falls back to files uploaded directly into /content.


In [ ]:
# 3. Safe File Loading Helpers

def resolve_input_path(manifest_path: str, fallback_filename: str) -> str:
    if manifest_path and os.path.exists(manifest_path):
        return manifest_path

    fallback_path = f"/content/{fallback_filename}"
    if os.path.exists(fallback_path):
        return fallback_path

    raise FileNotFoundError(
        f"Could not find input file. Tried: {manifest_path} and {fallback_path}"
    )

column_profile_path = resolve_input_path(
    manifest.get("column_profile_path"),
    "column_profile.csv"
)

missingness_summary_path = resolve_input_path(
    manifest.get("missingness_summary_path"),
    "missingness_summary.csv"
)

candidate_drop_review_path = resolve_input_path(
    manifest.get("candidate_drop_review_path"),
    "candidate_drop_review.csv"
)

profile = pd.read_csv(column_profile_path)
missing = pd.read_csv(missingness_summary_path)
decisions = pd.read_csv(candidate_drop_review_path)

print("Loaded:")
print(" -", column_profile_path)
print(" -", missingness_summary_path)
print(" -", candidate_drop_review_path)


Loaded:
 - /content/column_profile.csv
 - /content/missingness_summary.csv
 - /content/candidate_drop_review.csv



## Determine which columns to load from the parquet

Key to RAM saving  
Instead of loading all columns, the notebook uses candidate_drop_review.csv to load only fields marked:

- keep
- transform
- review

If those labels are missing, it falls back to all columns listed in
##column_profile.csv.


In [ ]:
# 4. Determine Columns to Keep

decisions.columns = [c.strip() for c in decisions.columns]
profile.columns = [c.strip() for c in profile.columns]

required_decision_cols = {"column_name", "recommended_action"}
if required_decision_cols.issubset(set(decisions.columns)):
    keep_cols = (
        decisions.loc[
            decisions["recommended_action"].astype(str).str.lower().isin(["keep", "transform", "review"]),
            "column_name"
        ]
        .dropna()
        .astype(str)
        .tolist()
    )
else:
    keep_cols = []

if not keep_cols:
    if "column_name" in profile.columns:
        keep_cols = profile["column_name"].dropna().astype(str).tolist()
    else:
        raise ValueError("Unable to determine keep columns from Notebook 1 outputs.")

keep_cols = list(dict.fromkeys(keep_cols))
print("Columns selected for load:", len(keep_cols))
print(keep_cols[:15], "..." if len(keep_cols) > 15 else "")


Columns selected for load: 36
['descriptor_2', 'landmark', 'intersection_street_1', 'intersection_street_2', 'cross_street_2', 'cross_street_1', 'address_type', 'location_type', 'city', 'street_name', 'resolution_description', 'descriptor', 'council_district', 'source_file', 'source_parquet'] ...



## Load only the selected columns from the raw parquet

This minimizes memory usage in Colab and keeps the conditioning step practical for the larger dataset.

## Add in

In [ ]:
import os
import json
import shutil
import warnings
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# 5. Load Raw Data with Selected Columns Only
#    Standard Colab RAM safe mode for 20M rows

# Standard Colab cannot reliably load ~20M rows into one pandas dataframe.
# This setting keeps the rest of the notebook runnable by loading a capped
# working sample from the full parquet file.
#
# IMPORTANT:
# - SOURCE_PARQUET still comes from run_manifest.json.
# - Full 20M aggregation should happen later in Notebook 3 using batch processing.

STANDARD_COLAB_SAMPLE_MODE = True
MAX_ROWS_TO_LOAD = 300_000        # Start here for standard Colab. Lower to 150_000 if RAM crashes.
PARQUET_BATCH_SIZE = 50_000       # Batch size used while reading from parquet.

# Confirm the manifest parquet path exists before reading.
# If this fails, the issue is Drive path/location, not the loading logic.
if not os.path.exists(SOURCE_PARQUET):
    raise FileNotFoundError(
        f"""The source parquet path from run_manifest.json was not found.\n\n"\
        f"SOURCE_PARQUET = {SOURCE_PARQUET}\n\n"\
        "Fix options:\n"\
        "1. Confirm Google Drive is mounted.\n"\
        "2. Confirm the parquet file exists at the path above.\n"\
        "3. If the file moved, update source_parquet inside run_manifest.json, then rerun."""
    )


def load_parquet_sample(path, columns, max_rows=300_000, batch_size=50_000):
    """Load a capped sample from a parquet file without reading all rows into RAM.

    This preserves the downstream df variable while avoiding a full pandas load.
    """
    pf = pq.ParquetFile(path)
    batches = []
    rows_loaded = 0

    for batch in pf.iter_batches(batch_size=batch_size, columns=columns):
        part = batch.to_pandas()
        remaining = max_rows - rows_loaded

        if len(part) > remaining:
            part = part.head(remaining)

        batches.append(part)
        rows_loaded += len(part)

        print(f"Loaded {rows_loaded:,} rows so far...")

        del batch, part
        gc.collect()

        if rows_loaded >= max_rows:
            break

    if not batches:
        raise RuntimeError("No rows were loaded from the parquet file.")

    out = pd.concat(batches, ignore_index=True)
    del batches
    gc.collect()
    return out


df = load_parquet_sample(
    SOURCE_PARQUET,
    columns=keep_cols,
    max_rows=MAX_ROWS_TO_LOAD,
    batch_size=PARQUET_BATCH_SIZE
)

print("Loaded dataframe shape:", df.shape)
print("NOTE: Standard Colab sample mode is enabled. This is a capped working sample, not the full 20M-row dataset.")
display(df.head())

Loaded 50,000 rows so far...
Loaded 100,000 rows so far...
Loaded 150,000 rows so far...
Loaded 200,000 rows so far...
Loaded 250,000 rows so far...
Loaded 300,000 rows so far...
Loaded dataframe shape: (300000, 36)
NOTE: Standard Colab sample mode is enabled. This is a capped working sample, not the full 20M-row dataset.


,descriptor_2,landmark,intersection_street_1,intersection_street_2,cross_street_2,cross_street_1,address_type,location_type,city,street_name,...,incident_address,closed_date,latitude,longitude,x_coordinate_state_plane,y_coordinate_state_plane,incident_zip,resolution_action_updated_date,created_date,unique_key
0,<NA>,<NA>,56 ROAD,RUST STREET,<NA>,<NA>,INTERSECTION,<NA>,QUEENS,<NA>,...,<NA>,<NA>,40.72444285430812,-73.91510543889461,1007781.0,203222.0,11378.0,2026-04-04T02:55:19.000,2026-04-04T02:55:19.000,68555060
1,<NA>,<NA>,59 STREET,QUEENS BOULEVARD,<NA>,<NA>,INTERSECTION,<NA>,QUEENS,<NA>,...,<NA>,2026-04-04T02:54:42.000,40.74171428915491,-73.90633230529538,1010206.0,209517.0,11377.0,2026-04-04T02:54:42.000,2026-04-04T02:54:42.000,68557159
2,<NA>,<NA>,101 AVENUE,WOODHAVEN BOULEVARD,<NA>,<NA>,INTERSECTION,<NA>,QUEENS,<NA>,...,<NA>,<NA>,40.68424002310187,-73.84609489919268,1026935.0,188601.0,11416.0,2026-04-04T02:25:42.000,2026-04-04T02:25:42.000,68549835
3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,ADDRESS,Mixed Use,NEW YORK,EAST 10 STREET,...,241 EAST 10 STREET,<NA>,40.728960833715206,-73.9848643587161,988445.0,204857.0,10003.0,2026-04-04T02:21:21.000,2026-04-04T02:21:21.000,68551639
4,<NA>,<NA>,GATES AVENUE,WYCKOFF AVENUE,<NA>,<NA>,INTERSECTION,<NA>,BROOKLYN,<NA>,...,<NA>,2026-04-04T02:01:53.000,40.6999378733718,-73.91180788979796,1008704.0,194295.0,11237.0,2026-04-04T02:01:53.000,2026-04-04T02:01:53.000,68551885



## Standardize column names

This creates analysis-safe column names for downstream modeling and reporting.


In [ ]:
# 6. Standardize Column Names

def clean_column_name(name: str) -> str:
    name = str(name).strip().lower()
    for old, new in [
        (" ", "_"),
        ("/", "_"),
        ("-", "_"),
        ("(", ""),
        (")", ""),
        (".", ""),
        (",", ""),
        ("&", "and"),
        ("__", "_"),
    ]:
        name = name.replace(old, new)
    while "__" in name:
        name = name.replace("__", "_")
    return name.strip("_")

original_columns = list(df.columns)
cleaned_columns = [clean_column_name(c) for c in original_columns]

column_name_mapping = pd.DataFrame({
    "original_column_name": original_columns,
    "cleaned_column_name": cleaned_columns
})

df.columns = cleaned_columns
column_name_mapping.to_csv(f"{TABLE_DIR}/column_name_mapping.csv", index=False)

print("Column names standardized.")
display(column_name_mapping.head())


Column names standardized.


,original_column_name,cleaned_column_name
0,descriptor_2,descriptor_2
1,landmark,landmark
2,intersection_street_1,intersection_street_1
3,intersection_street_2,intersection_street_2
4,cross_street_2,cross_street_2



## Standardize datetime fields where possible

This attempts datetime conversion on likely date columns only, which is safer for memory and avoids unnecessary work.


In [ ]:
# 7. Datetime Standardization

datetime_candidates = [
    c for c in df.columns
    if any(token in c for token in ["date", "time", "created", "closed", "opened", "updated"])
]

datetime_conversion_log = []

for col in datetime_candidates:
    try:
        before_nonnull = df[col].notna().sum()
        converted = pd.to_datetime(df[col], errors="coerce")
        after_nonnull = converted.notna().sum()
        if after_nonnull > 0:
            df[col] = converted
            status = "converted"
        else:
            status = "not_converted"
        datetime_conversion_log.append({
            "column_name": col,
            "status": status,
            "before_nonnull": int(before_nonnull),
            "after_nonnull": int(after_nonnull)
        })
    except Exception:
        datetime_conversion_log.append({
            "column_name": col,
            "status": "error",
            "before_nonnull": int(df[col].notna().sum()),
            "after_nonnull": 0
        })

pd.DataFrame(datetime_conversion_log).to_csv(f"{TABLE_DIR}/datetime_conversion_log.csv", index=False)
print("Datetime conversion attempted for", len(datetime_candidates), "columns.")


Datetime conversion attempted for 3 columns.



## Handle missing values

Missing values are treated based on data type:

- categorical/text fields --> "UNKNOWN"
- numeric fields --> median
- datetime fields --> left as-is unless specifically engineered later

This step also records the treatment used for each field.


In [ ]:
# 8. Missing Value Handling

missing_value_treatment = []

for col in df.columns:
    series = df[col]
    null_pct = float(series.isna().mean())

    if pd.api.types.is_datetime64_any_dtype(series):
        action = "leave_datetime_nulls"
    elif pd.api.types.is_numeric_dtype(series):
        median_val = series.median()
        if pd.isna(median_val):
            action = "all_null_numeric_leave"
        else:
            df[col] = series.fillna(median_val)
            action = "numeric_median_impute"
    else:
        df[col] = series.fillna("UNKNOWN")
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()
        action = "categorical_fill_unknown"

    missing_value_treatment.append({
        "column_name": col,
        "null_pct_before": null_pct,
        "treatment": action
    })

missing_treatment_df = pd.DataFrame(missing_value_treatment)
missing_treatment_df.to_csv(f"{TABLE_DIR}/missing_value_treatment.csv", index=False)

print("Missing value treatment complete.")
display(missing_treatment_df.head())


Missing value treatment complete.


,column_name,null_pct_before,treatment
0,descriptor_2,0.582300,categorical_fill_unknown
1,landmark,0.473277,categorical_fill_unknown
2,intersection_street_1,0.392790,categorical_fill_unknown
3,intersection_street_2,0.392430,categorical_fill_unknown
4,cross_street_2,0.352617,categorical_fill_unknown



## Clean categorical text fields

This reduces inconsistency such as extra spaces and mixed-case labels in object columns.


In [ ]:
# 9. Clean Categorical Text Fields

object_cols = df.select_dtypes(include=["object"]).columns.tolist()

for col in object_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .replace({"": "UNKNOWN", "nan": "UNKNOWN", "None": "UNKNOWN"})
    )

print("Categorical text cleaning complete for", len(object_cols), "columns.")


Categorical text cleaning complete for 0 columns.



## Feature engineering

This is where raw fields are turned into modeling-ready variables.

Key engineered fields:
- response_time_hours
- closed_same_day_flag
- repeat_complaint_flag

These are the features that make the downstream modeling stage useful.


In [ ]:

# 10. Feature Engineering

derived_features = []

# Response time in hours
created_candidates = [c for c in df.columns if c in ["created_date", "created_time", "created"]]
closed_candidates = [c for c in df.columns if c in ["closed_date", "closed_time", "closed"]]

created_col = created_candidates[0] if created_candidates else None
closed_col = closed_candidates[0] if closed_candidates else None

if created_col and closed_col:
    df["response_time_hours"] = (df[closed_col] - df[created_col]).dt.total_seconds() / 3600
    derived_features.append({
        "derived_feature": "response_time_hours",
        "source_columns": f"{created_col}, {closed_col}",
        "purpose": "Regression target for service response time"
    })

    df["closed_same_day_flag"] = (
        (df[closed_col].dt.date == df[created_col].dt.date) &
        df[closed_col].notna() &
        df[created_col].notna()
    ).astype(int)
    derived_features.append({
        "derived_feature": "closed_same_day_flag",
        "source_columns": f"{created_col}, {closed_col}",
        "purpose": "Classification target for same-day resolution"
    })

# Repeat complaint proxy
complaint_candidates = [c for c in df.columns if c in ["complaint_type", "complainttype"]]
if complaint_candidates:
    complaint_col = complaint_candidates[0]
    value_counts = df[complaint_col].value_counts(dropna=False)
    repeated_values = set(value_counts[value_counts > 1].index.tolist())
    df["repeat_complaint_flag"] = df[complaint_col].isin(repeated_values).astype(int)
    derived_features.append({
        "derived_feature": "repeat_complaint_flag",
        "source_columns": complaint_col,
        "purpose": "Exploratory classification flag for repeated complaint categories"
    })

derived_features_df = pd.DataFrame(derived_features)
derived_features_df.to_csv(f"{TABLE_DIR}/derived_features_summary.csv", index=False)

print("Feature engineering complete.")
display(derived_features_df)


Feature engineering complete.


,derived_feature,source_columns,purpose
0,response_time_hours,"created_date, closed_date",Regression target for service response time
1,closed_same_day_flag,"created_date, closed_date",Classification target for same-day resolution
2,repeat_complaint_flag,complaint_type,Exploratory classification flag for repeated c...



## Final field list and transformation log

This documents what the final prepared dataset contains and what transformations were applied.


In [ ]:
# 11. Save Final Field List and Transformation Rules

final_field_list = pd.DataFrame({
    "column_name": df.columns
})
final_field_list.to_csv(f"{TABLE_DIR}/final_field_list.csv", index=False)

transformation_rules_applied = pd.DataFrame([
    {"step": "column_selection", "details": "Loaded only columns marked keep/transform/review from candidate_drop_review.csv"},
    {"step": "column_name_standardization", "details": "Converted source columns into analysis-safe lowercase underscore names"},
    {"step": "datetime_conversion", "details": "Attempted datetime conversion on likely datetime columns"},
    {"step": "missing_value_treatment", "details": "Categorical fields filled with UNKNOWN; numeric fields median-imputed; datetime nulls retained"},
    {"step": "categorical_text_cleaning", "details": "Trimmed and normalized text in object columns"},
    {"step": "feature_engineering", "details": "Created response_time_hours, closed_same_day_flag, and repeat_complaint_flag where source columns were available"}
])
transformation_rules_applied.to_csv(f"{TABLE_DIR}/transformation_rules_applied.csv", index=False)

print("Saved final_field_list.csv and transformation_rules_applied.csv")


Saved final_field_list.csv and transformation_rules_applied.csv



## Save the prepared datasets

This writes:
- a full prepared parquet to Google Drive
- a smaller local sample parquet for Notebook 3

The sample keeps Notebook 3 practical in Colab.


In [ ]:
# 12. Save Prepared Datasets

sample_n = min(150000, len(df))
analysis_ready_sample = df.sample(n=sample_n, random_state=42) if len(df) > sample_n else df.copy()

analysis_ready_sample_path = f"{COND_DIR}/analysis_ready_sample.parquet"
analysis_ready_sample.to_parquet(analysis_ready_sample_path, index=False)

# Keep the same output variable/name for downstream compatibility.
# In standard Colab sample mode, this file contains the prepared working sample,
# not the full 20M-row prepared dataset.
analysis_ready_full_path = "/content/drive/MyDrive/project_data/analysis_ready_full.parquet"
df.to_parquet(analysis_ready_full_path, index=False)

print("Saved local sample parquet:", analysis_ready_sample_path)
print("Saved prepared working dataset to Drive:", analysis_ready_full_path)
if 'STANDARD_COLAB_SAMPLE_MODE' in globals() and STANDARD_COLAB_SAMPLE_MODE:
    print("NOTE: analysis_ready_full.parquet contains the capped working sample because Standard Colab sample mode is enabled.")


Saved local sample parquet: /content/project_outputs/02_conditioning/analysis_ready_sample.parquet
Saved prepared working dataset to Drive: /content/drive/MyDrive/project_data/analysis_ready_full.parquet
NOTE: analysis_ready_full.parquet contains the capped working sample because Standard Colab sample mode is enabled.



## Update manifest for Notebook 3

This records the Notebook 2 outputs and the final prepared dataset location.


In [ ]:
# 13. Update Manifest for Notebook 3

manifest_out = dict(manifest)
manifest_out["notebook_stage"] = "Notebook 2 - Conditioning and Feature Finalization"
manifest_out["conditioning_dir"] = COND_DIR
manifest_out["analysis_ready_sample_path"] = analysis_ready_sample_path
manifest_out["analysis_ready_full_path"] = analysis_ready_full_path
manifest_out["final_field_list_path"] = f"{TABLE_DIR}/final_field_list.csv"
manifest_out["missing_value_treatment_path"] = f"{TABLE_DIR}/missing_value_treatment.csv"
manifest_out["derived_features_summary_path"] = f"{TABLE_DIR}/derived_features_summary.csv"
manifest_out["transformation_rules_applied_path"] = f"{TABLE_DIR}/transformation_rules_applied.csv"
manifest_out["column_name_mapping_path"] = f"{TABLE_DIR}/column_name_mapping.csv"
manifest_out["standard_colab_sample_mode"] = bool(globals().get("STANDARD_COLAB_SAMPLE_MODE", False))
manifest_out["max_rows_loaded_notebook2"] = int(globals().get("MAX_ROWS_TO_LOAD", len(df)))
manifest_out["notebook2_full_data_note"] = "In standard Colab sample mode, analysis_ready_full_path contains the prepared working sample, not the full 20M rows. Notebook 3 should use source_parquet for full batch aggregations and analysis_ready_sample_path for modeling."

manifest_out["handoff_to_notebook_3"] = {
    "primary_files": [
        manifest_out["final_field_list_path"],
        manifest_out["missing_value_treatment_path"],
        manifest_out["derived_features_summary_path"],
        manifest_out["transformation_rules_applied_path"],
        manifest_out["analysis_ready_sample_path"],
    ],
    "large_file": analysis_ready_full_path,
    "purpose": "Use these files to run statistical analysis and modeling in Notebook 3."
}

manifest_out_path = "/content/run_manifest_notebook2.json"
with open(manifest_out_path, "w") as f:
    json.dump(manifest_out, f, indent=2)

print("Updated Notebook 2 manifest:", manifest_out_path)


Updated Notebook 2 manifest: /content/run_manifest_notebook2.json


In [ ]:
zip_dir = "/content/notebook2_handoff"
os.makedirs(zip_dir, exist_ok=True)

for src in [
    f"{TABLE_DIR}/final_field_list.csv",
    f"{TABLE_DIR}/missing_value_treatment.csv",
    f"{TABLE_DIR}/derived_features_summary.csv",
    f"{TABLE_DIR}/transformation_rules_applied.csv",
    analysis_ready_sample_path,
    manifest_out_path,
    completion_summary_path,
]:
    shutil.copy(src, os.path.join(zip_dir, os.path.basename(src)))

zip_path = "/content/notebook2_outputs.zip"
shutil.make_archive("/content/notebook2_outputs", "zip", zip_dir)

print("Created:", zip_path)